In [1]:
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import CreateTable, CreateTableColumn, CreateTableConstraints, CreateForeignKey
import pandas as pd
from pandas.core.interchange.dataframe_protocol import DataFrame
from dotenv import load_dotenv
import os 

load_dotenv()
password = os.getenv("DBREPO_PASS")
username = os.getenv("DBREPO_USER")
client = RestClient("https://test.dbrepo.tuwien.ac.at/", username=username, password=password)

containers = client.get_containers()
print(containers)

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [2]:
df = client.get_database("5cde660e-153a-4bff-8e41-69e87cda399d")

## Check all views

In [3]:
for t in df.views:
    print(t.name, t.id)

drug_gdp_features_view 6a6080f4-4117-4201-af05-876bf9eb05d5
ww_city_year_drug_summary 8550c148-db32-475c-8be6-d55e34782949


## Import view from API

In [4]:
db_id = "5cde660e-153a-4bff-8e41-69e87cda399d"
view_id = "6a6080f4-4117-4201-af05-876bf9eb05d5"

response = client._wrapper(
    method="get",
    url=f"/api/v1/database/{db_id}/view/{view_id}"
)

print(response.status_code)
db = response.json()

200


In [5]:
print(db)

{'id': '6a6080f4-4117-4201-af05-876bf9eb05d5', 'name': 'drug_gdp_features_view', 'identifiers': [], 'query': 'select `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`daily_mean_concentration` as `daily_mean`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`metabolite_name` as `metabolite_name`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`ref_year` as `ref_year`, `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code` as `nuts_code`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` as `city_name`, `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`gdp` as `gdp` from `wastewater_data` join `city_map` on `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`city_name` join `gdp_data` on `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`nuts_code` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code`', 'owner': {'id': None, 'username': 'data_st

In [6]:
response = client._wrapper(
    method="get",
    url=f"/api/v1/database/{db_id}/view/{view_id}/data",
    headers={"Accept": "application/json"}
)
print(response.status_code)
print(response.json())

200
[{'city_name': 'Purgstall', 'daily_mean': 22.03, 'gdp': 6574580000.0, 'metabolite_name': 'cocaine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 4.64, 'gdp': 6574580000.0, 'metabolite_name': 'MDMA', 'nuts_code': 'AT121', 'ref_year': 2020}, {'city_name': 'Purgstall', 'daily_mean': 13.74, 'gdp': 6574580000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 1.48, 'gdp': 6574580000.0, 'metabolite_name': 'methamphetamine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 30.6, 'gdp': 6574580000.0, 'metabolite_name': 'cannabis', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 24.39, 'gdp': 6574580000.0, 'metabolite_name': 'cocaine', 'nuts_code': 'AT121', 'ref_year': 2020}, {'city_name': 'Purgstall', 'daily_mean': 1.32, 'gdp': 6574580000.0, 'metabolite_name': 'methamphetamine', 'nuts_code': 'AT121', 'ref_year': 2020}, {

## Transform into dataset, ready to be used

In [7]:
import pandas as pd
df = pd.DataFrame(response.json())

In [8]:
print(df)

   city_name  daily_mean           gdp  metabolite_name nuts_code  ref_year
0  Purgstall       22.03  6.574580e+09          cocaine     AT121      2019
1  Purgstall        4.64  6.574580e+09             MDMA     AT121      2020
2  Purgstall       13.74  6.574580e+09      amphetamine     AT121      2019
3  Purgstall        1.48  6.574580e+09  methamphetamine     AT121      2019
4  Purgstall       30.60  6.574580e+09         cannabis     AT121      2019
5  Purgstall       24.39  6.574580e+09          cocaine     AT121      2020
6  Purgstall        1.32  6.574580e+09  methamphetamine     AT121      2020
7  Purgstall        4.41  6.574580e+09             MDMA     AT121      2019
8  Purgstall       23.77  6.574580e+09      amphetamine     AT121      2020
9  Purgstall       38.96  6.574580e+09         cannabis     AT121      2020


## Test if results stay the same

In [9]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLM


In [10]:
final_data = df.copy()

In [11]:
print(final_data.columns)
#city	year	metabolite_name	daily_mean_concentration	nuts_code	gdp


Index(['city_name', 'daily_mean', 'gdp', 'metabolite_name', 'nuts_code',
       'ref_year'],
      dtype='object')


In [12]:
final_data = final_data.rename(columns={"city_name": "city", "ref_year": "year", "daily_mean" : "daily_mean_concentration"})

In [13]:
print(final_data.columns)

Index(['city', 'daily_mean_concentration', 'gdp', 'metabolite_name',
       'nuts_code', 'year'],
      dtype='object')


In [14]:
print(final_data.shape)

(10, 6)


In [15]:
print(final_data)

        city  daily_mean_concentration           gdp  metabolite_name  \
0  Purgstall                     22.03  6.574580e+09          cocaine   
1  Purgstall                      4.64  6.574580e+09             MDMA   
2  Purgstall                     13.74  6.574580e+09      amphetamine   
3  Purgstall                      1.48  6.574580e+09  methamphetamine   
4  Purgstall                     30.60  6.574580e+09         cannabis   
5  Purgstall                     24.39  6.574580e+09          cocaine   
6  Purgstall                      1.32  6.574580e+09  methamphetamine   
7  Purgstall                      4.41  6.574580e+09             MDMA   
8  Purgstall                     23.77  6.574580e+09      amphetamine   
9  Purgstall                     38.96  6.574580e+09         cannabis   

  nuts_code  year  
0     AT121  2019  
1     AT121  2020  
2     AT121  2019  
3     AT121  2019  
4     AT121  2019  
5     AT121  2020  
6     AT121  2020  
7     AT121  2019  
8     AT121  202